# Average Multiple Days Together

## Imports and Versions

In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from typing import Literal
from astropy import units as u
from scipy.stats import norm

from pygsdata import GSData, plots, GSFlag
from pygsdata.register import gsregister

from edges import modeling as mdl
from edges.averaging import average_over_times, NsamplesStrategy, get_weights_from_strategy
from edges.analysis.datamodel import add_model
from edges.filters import rfi_model_filter
from edges.alanmode import read_spec_txt
from edges.filters import filters
from edges.filters.filters import gsdata_filter
from edges.averaging import averaging

from edges_pipeline_utils import utils


In [2]:
plt.style.use("default")

In [3]:
utils.print_versions()


Versions: 
            read_acq: 1.2.0
            pygsdata: 0.2.3
      edges-analysis: 8.0.1.dev10+gc4754b807


## Parameters and Data Loading

In [4]:
gathered_days_file: str = "gathered-days.gsh5"
alandir: str = "/home/smurray/data4/edges/alans-pipeline/scripts/H2CaseFieldData/"

do_rfi_round_2: bool = True
do_rfi_round_3: bool = True
raise_unmatching: bool = True
compare_alan: bool = True

In [5]:
# Parameters
raise_unmatching = False
compare_alan = True
gathered_days_file = "/data7/smurray/edges/projects_with_nive/edges-bowman2018-pipeline/work/0f/cccdf7d99544c451918a6b6f1479af/gathered-days.gsh5"


In [6]:
gathered_days_file = Path(gathered_days_file)

In [7]:
data = GSData.from_file(gathered_days_file)

## Analysis and Averaging

### Model the spectra so we can average residuals only

When we perform the average over nights, we average the models and residuals separately (and the residuals receive frequency-dependent weights while the models do not). First, we model each night here.

In [8]:
data = add_model(data, model=mdl.models.PhysicalIono(spectral_index=-2.55, f_center=75.0, n_terms=5), nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM)

In [9]:
alanmodel = np.genfromtxt("/home/smurray/data4/edges/alans-pipeline/scripts/nightly_model_file.txt")

In [10]:
alan_dates = np.genfromtxt("/home/smurray/data4/edges/alans-pipeline/scripts/nightly_dates.txt")

In [11]:
alan_data = np.genfromtxt("/home/smurray/data4/edges/alans-pipeline/scripts/nightly_data.txt")

In [12]:
for i in range(data.ntimes):
    plt.plot(data.freqs, data.model[0,0,i] - alanmodel[i])
plt.xlabel("Frequency [MHz]")
plt.ylabel("Model Difference [K]")
plt.title("Difference in night-to-night models between edges-analysis and C-code")

Text(0.5, 1.0, 'Difference in night-to-night models between edges-analysis and C-code')

### Some Filters

Our first filter is an RMS filter (just thresholding a whole night based on its RMS to the fitted model)

In [13]:
alanfilt = np.genfromtxt("/home/smurray/data4/edges/alans-pipeline/scripts/nightly_flagfile.txt")

In [14]:
alflags = GSFlag(~(alanfilt[:, 1].astype(bool)), axes=("time",))

In [15]:
rms_data = filters.rms_filter(data, threshold=0.17, nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM) 
rms_data_alan =  data.add_flags(flags=alflags, filt='applying Alans RMS filter flags')

In [16]:
if raise_unmatching:
    assert np.sum(~rms_data.flags['rms_filter'].flags[0,0] ^ alanfilt[:, 1].astype(bool))==0
elif compare_alan:
    print("Unmatched flags: ", np.sum(~rms_data.flags['rms_filter'].flags[0,0] ^ alanfilt[:, 1].astype(bool)))

Unmatched flags:  0


In [17]:
rms_data = filters.prune_flagged_integrations(rms_data)
rms_data_alan = filters.prune_flagged_integrations(rms_data_alan)

In [18]:
print(rms_data.ntimes, rms_data_alan.ntimes)

62 62


Based on the model already fit, we perform simple RFI-flagging, where we flag any channel whose residual is larger than a threshold multiplied by the RMS of the residuals that night (over frequency).

In [19]:
rms_data.write_gsh5("filtered-gathered-data.gsh5")

GSData(telescope=Telescope(name='edges-low-alan', location=<EarthLocation (-2544175.08852437, 5102825.79106231, -2848558.83597113) m>, pols=('XX',), integration_time=<Quantity 13. s>, x_orientation=<Angle 0. deg>), data=array([[[[4932.97079161, 4837.00838405, 4744.2497104 , ...,
           871.1798623 ,  862.75287787,  853.61268599],
         [4951.89555586, 4855.85942192, 4762.25964463, ...,
           873.46992828,  865.13659105,  856.50861185],
         [4942.15482362, 4845.74815362, 4752.55807579, ...,
           872.29190505,  863.8423989 ,  854.68095892],
         ...,
         [4681.89952764, 4589.87408775, 4500.36747517, ...,
           807.38301554,  799.30684478,  790.41574023],
         [4971.2495432 , 4874.07620671, 4777.96309517, ...,
           867.03652659,  858.16633326,  849.80262549],
         [4962.83235666, 4866.93042795, 4769.78776188, ...,
           865.56360784,  856.61946002,  847.91228104]]]],
      shape=(1, 1, 62, 128)), freqs=<Quantity [50.01220703, 50.4028

In [20]:
if do_rfi_round_2:
    filt_data = filters.rms_rfi_filter(rms_data, threshold=1.9, nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM)
    filt_data_alan = filters.rms_rfi_filter(rms_data_alan, threshold=1.9)
else:
    filt_data = rms_data
    filt_data_alan = rms_data_alan

Now, make sure that we flagged exactly the same channels/nights as the C-code

In [21]:
alanweights= np.genfromtxt("/home/smurray/data4/edges/alans-pipeline/scripts/nightly_weights.txt")
weights= filt_data.flagged_nsamples[0,0] > 0

if raise_unmatching and do_rfi_round_2:
    assert np.sum(weights ^ alanweights.astype(bool))==0
elif compare_alan:
    print("Number of flags differing: ", np.sum(weights ^ alanweights.astype(bool)))

Number of flags differing:  2


In [22]:
plots.plot_waterfall(filt_data);

### Inspect the noise distribution at this stage

In [23]:
nsamples_strategy = NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM
w = get_weights_from_strategy(rms_data, nsamples_strategy)[0]
rms = np.sqrt(np.average(np.square(rms_data.residuals), weights=w, axis=-1))[..., None]

nflagged = np.nansum(np.where(rms_data.flagged_nsamples>0, (rms_data.residuals/rms), -np.inf).flatten()>1.9)
ntot = np.nansum(np.where(rms_data.flagged_nsamples>0, (rms_data.residuals/rms), -np.inf).flatten()>-20)

In [24]:
plt.hist(np.where(rms_data.flagged_nsamples>0, (rms_data.residuals/rms), np.nan).flatten(), bins=70, density=True);
x = np.linspace(-4, 4, 101)
plt.plot(x, norm.pdf(x))
plt.axvline(1.9, color='k', ls='--')
plt.xlabel("Z Score")
plt.ylabel("PDF")
plt.text(2, 0.4, r"$1.9\sigma$")
plt.text(2.5, 0.05, rf"{nflagged*100/ntot:.1}\% \\ data cut")

Text(2.5, 0.05, '3e+00\\% \\\\ data cut')

### Average all nights

In [25]:
avg_data = average_over_times(
    filt_data, use_resids=True, nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM
)
avg_data_alan = average_over_times(
    filt_data_alan, use_resids=True, nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM
)

/data7/smurray/edges/projects_with_nive/edges-bowman2018-pipeline/.venv/lib/python3.11/site-packages/edges/averaging/lstbin.py:101: RuntimeWarning: invalid value encountered in divide
  mean_resids = sum_resids / ntot


### Last RFI Filter

In [26]:
avg_data = add_model(avg_data, model=mdl.Polynomial(offset=-2.5, n_terms=7, transform=mdl.ScaleTransform(scale=75.0)), nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES_UNIFORM)
avg_data_alan = add_model(avg_data_alan, model=mdl.Polynomial(offset=-2.5, n_terms=7, transform=mdl.ScaleTransform(scale=75.0)), nsamples_strategy=NsamplesStrategy.FLAGGED_NSAMPLES)

In [27]:
if do_rfi_round_3:
    final_data = filters.rms_rfi_filter(avg_data, threshold=1.9)
    final_data_alan = filters.rms_rfi_filter(avg_data_alan, threshold=1.9)
else:
    final_data = avg_data
    final_data_alan = avg_data_alan

In [28]:
final_data = add_model(final_data, model=mdl.LinLog(n_terms=5))
final_data_alan = add_model(final_data_alan, model=mdl.LinLog(n_terms=5))

## Inspect Final Results

In [29]:
alan_fl = f"{alandir}/final_average_latest.txt"
alan = np.genfromtxt(alan_fl, usecols=(1, 3, 6, 9, 12), names=('freq', 'tant', 'model', 'resid', 'weight'))

In [30]:
linlog = mdl.LinLog(n_terms=5).at(x=final_data.freqs)

In [31]:
ourfit = linlog.fit(ydata=final_data.data[0,0,0], weights=final_data.flagged_nsamples[0,0,0]>0)
ourfit_alan = linlog.fit(ydata=final_data_alan.data[0,0,0], weights=final_data_alan.flagged_nsamples[0,0,0]>0)
alfit = linlog.fit(ydata=alan['tant'], weights=alan['weight'])

In [32]:
def plot_final_data(final_data, label: str):
    fig, ax = plt.subplots(3, 1, sharex=True, figsize=(10, 8), constrained_layout=True)

    ax[0].plot(final_data.freqs, np.where(final_data.flagged_nsamples[0,0,0]>0, final_data.data[0,0,0], np.nan), label=label)
    ax[0].plot(final_data.freqs, np.where(alan['weight']>0, alan['tant'], np.nan), label='C-code', ls='--')
    ax[0].legend()
    ax[0].text(0.95, 0.9, "Spectrum", transform=ax[0].transAxes, ha='right', fontweight='bold')

    for i in range(3):
        ax[i].set_ylabel("Temperature [K]")
        
    ttmin = data.times.min().datetime.timetuple()
    ttmax = data.times.max().datetime.timetuple()

    fig.suptitle(f"Final Averaged Spectrum: {rms_data.ntimes} days [{ttmin.tm_year}:{ttmin.tm_yday:>03} -- {ttmax.tm_year}:{ttmax.tm_yday:>03}]")

    ax[1].plot(final_data.freqs, np.where(final_data.flagged_nsamples[0,0,0]>0, ourfit.residual, np.nan))

    ax[1].plot(final_data.freqs, np.where(final_data.flagged_nsamples[0,0,0]>0, alfit.residual, np.nan), ls='--')
    ax[1].text(0.95, 0.9, "FG Residuals (LinLog, 5-term)", transform=ax[1].transAxes, ha='right', fontweight='bold')


    ax[2].plot(final_data.freqs, np.where(final_data.flagged_nsamples[0,0,0]>0, final_data.data[0,0,0] - alan['tant'], np.nan), color='b')
    ax[2].set_xlabel("Frequency [MHz]")
    ax[2].text(0.95, 0.9, "Absolute Difference between pipelines", transform=ax[2].transAxes, ha='right', fontweight='bold');


In [33]:
plot_final_data(final_data, 'edges-analysis')

In [34]:
plot_final_data(final_data_alan, 'edges-analysis w/ final nights from B18')

We ensure that we have all the same flags as the C-code:

In [35]:
if raise_unmatching:
    assert np.sum((final_data.flagged_nsamples > 0) ^ alan['weight'].astype(bool))==0
elif compare_alan:
    print("Unmatched flags: ", np.sum((final_data.flagged_nsamples > 0) ^ alan['weight'].astype(bool)))

Unmatched flags:  0


In [36]:
plt.plot(final_data.freqs, final_data.flagged_nsamples[0,0,0])
plt.xlabel("Frequency [MHz]")
plt.ylabel("Nsamples");

## Write out the data

In [37]:
final_data.write_gsh5(gathered_days_file.parent / "averaged_spectrum.gsh5");
final_data_alan.write_gsh5(gathered_days_file.parent / "averaged_spectrum_legacy_days.gsh5")

GSData(telescope=Telescope(name='edges-low-alan', location=<EarthLocation (-2544175.08852437, 5102825.79106231, -2848558.83597113) m>, pols=('XX',), integration_time=<Quantity 13. s>, x_orientation=<Angle 0. deg>), data=array([[[[   0.        ,    0.        ,    0.        , 4645.46071855,
          4556.499486  , 4469.79130911, 4385.32138728, 4303.11903117,
          4223.01051135, 4144.99370915, 4068.95764044, 3994.77396279,
          3922.60173032, 3852.17004072, 3783.43836165, 3716.41751808,
          3651.0423327 , 3587.21628274, 3525.00506709, 3464.25494044,
          3404.86318892, 3346.94369596, 3290.36327165, 3235.08405186,
          3181.10309182, 3128.37127843, 3076.86286409, 3026.48163128,
          2977.25054723, 2929.12776105, 2882.05583961, 2836.03603487,
          2791.01738184, 2746.9871241 , 2703.87501175, 2661.70206463,
          2620.39865799, 2580.0265294 , 2540.51935387, 2501.78205591,
          2463.90524794, 2426.7620429 , 2390.40791464, 2354.78081414,
          